In [1]:
# notebook to spot check the output of the updated phenocopy analysis:
# should have all variants with an allele frequency in gnomAD that are labeled as:
# greater than the lead variant (rare, larger effect, vars of interest)
# lower than the lead variant (non-emVar, no effect, control vars)

In [13]:
# import packages
import pandas as pd
import random
from collections import Counter
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', None)

In [2]:
# open variants file
shadowVars = pd.read_csv('results_final/highPIP_shadow_variants.tsv',
                         sep = '\t')

In [3]:
shadowVars.head()

,variant_id,chrom,pos,ref,alt,cell_type,skew_pred,abs_skew,enhancer_ids,af,AF_afr,AF_ami,AF_amr,AF_asj,AF_eas,AF_fin,AF_mid,AF_nfe,AF_remaining,AF_sas,exceeds_lead_skew,is_lead_variant,lead_variant_id,lead_abs_skew,lead_pip,lead_tissue,phenotype_id,gene_name,biotype,af_gtex,afc,afc_se,cs_id,cs_size,variant_id_gtex,skew_fc_vs_lead,lead_skew_sign
0,chr1:235647517:A:C,chr1,235647517,A,C,k562,0.122555,0.122555,EH38E2875581,0.000007,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000015,0.000000,0.000000,False,False,chr1:235647538:G:A,0.769824,0.983101,Brain_Putamen_basal_ganglia,ENSG00000168243.11,GNG4,protein_coding,0.442688,-0.362746,0.135328,1,1,1_235647517_A_C_b38,0.159198,-1
1,chr1:235647519:C:T,chr1,235647519,C,T,k562,-0.063890,0.063890,EH38E2875581,0.000013,0.000048,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,False,False,chr1:235647538:G:A,0.769824,0.983101,Brain_Putamen_basal_ganglia,ENSG00000168243.11,GNG4,protein_coding,0.442688,-0.362746,0.135328,1,1,1_235647519_C_T_b38,0.082993,-1
2,chr1:235647531:G:A,chr1,235647531,G,A,k562,0.050092,0.050092,EH38E2875581,0.000013,0.000000,0.00000,0.000000,0.000576,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,False,False,chr1:235647538:G:A,0.769824,0.983101,Brain_Putamen_basal_ganglia,ENSG00000168243.11,GNG4,protein_coding,0.442688,-0.362746,0.135328,1,1,1_235647531_G_A_b38,0.065069,-1
3,chr1:235647537:C:T,chr1,235647537,C,T,k562,-0.522557,0.522557,EH38E2875581,0.000007,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000015,0.000000,0.000000,False,False,chr1:235647538:G:A,0.769824,0.983101,Brain_Putamen_basal_ganglia,ENSG00000168243.11,GNG4,protein_coding,0.442688,-0.362746,0.135328,1,1,1_235647537_C_T_b38,0.678801,-1
4,chr1:235647538:G:A,chr1,235647538,G,A,k562,-0.769824,0.769824,EH38E2875581,0.411064,0.370026,0.54057,0.330084,0.459389,0.077429,0.530438,0.428082,0.470156,0.407109,0.224855,False,True,chr1:235647538:G:A,0.769824,0.983101,Brain_Putamen_basal_ganglia,ENSG00000168243.11,GNG4,protein_coding,0.442688,-0.362746,0.135328,1,1,1_235647538_G_A_b38,1.000000,-1


In [44]:
len(shadowVars['enhancer_ids'].unique())

672

In [4]:
# define a function for returning a random enhancer id for spot checking ing gnomAD/GTEx
def pickAcardAnycard (vars_df,
                      cell_type):
    # filter for cell type
    cellVars = vars_df[vars_df['cell_type'] == cell_type.lower()].copy()
    # get length of enahncer ids for selecting a random enhancer
    nEnh = len(cellVars['enhancer_ids'].unique())
    # pick random ID
    randEnh = cellVars['enhancer_ids'].tolist()[random.randint(0,nEnh)]
    # filter for only that id and sort
    enhVars = cellVars[cellVars['enhancer_ids'] == randEnh].copy().sort_values(['is_lead_variant', 'exceeds_lead_skew', 'af'], ascending = False)
    return enhVars

In [42]:
# define summary function
def summarizeShadows (vars_df, 
                      cell_types):
    # iterate through cell types
    for cell_type in cell_types:
        # filter for cell type
        cellVars = vars_df[vars_df['cell_type'] == cell_type.lower()].copy()
        # get the number of lead variants
        print(
            f'#### {cell_type.upper()} SUMMARY ####\n'
            f'there are {len(cellVars['lead_variant_id'].unique())} lead variants that are predicted emVars\n'
            f'they are in {len(cellVars['enhancer_ids'].unique())} enhancers\n'
            f'they impact {len(cellVars['phenotype_id'].unique())} genes\n'
            f'there are {len(cellVars[cellVars['exceeds_lead_skew'] == True])} variants predicted to have greater skew than the lead variant\n'
            f'there are {len(cellVars[(cellVars['exceeds_lead_skew'] == False) & (cellVars['is_lead_variant'] == False)])} control variants'
        )

In [43]:
summarizeShadows(
    shadowVars,
    ['k562', 'hepg2', 'sknsh']
)

#### K562 SUMMARY ####
there are 453 lead variants that are predicted emVars
they are in 453 enhancers
they impact 440 genes
there are 633 variants predicted to have greater skew than the lead variant
there are 30441 control variants
#### HEPG2 SUMMARY ####
there are 351 lead variants that are predicted emVars
they are in 351 enhancers
they impact 344 genes
there are 341 variants predicted to have greater skew than the lead variant
there are 23784 control variants
#### SKNSH SUMMARY ####
there are 430 lead variants that are predicted emVars
they are in 430 enhancers
they impact 418 genes
there are 549 variants predicted to have greater skew than the lead variant
there are 29105 control variants


In [ ]:
# get a summary of the data
# k562

In [18]:
# get the number of enhancers per biotype
enhPerBioType = shadowVars.copy()
enhPerBioType.loc[:, 'count'] = [1 for i in range(len(enhPerBioType))]
# count enhancers for each biotype
enhPerBioType_groupbyCounts = enhPerBioType.filter(['enhancer_ids', 'biotype', 'count', 'cell_type']).groupby('enhancer_ids').sum()

In [11]:
pickAcardAnycard(
    shadowVars,
    'sknsh'
)

,variant_id,chrom,pos,ref,alt,cell_type,skew_pred,abs_skew,enhancer_ids,af,AF_afr,AF_ami,AF_amr,AF_asj,AF_eas,AF_fin,AF_mid,AF_nfe,AF_remaining,AF_sas,exceeds_lead_skew,is_lead_variant,lead_variant_id,lead_abs_skew,lead_pip,lead_tissue,phenotype_id,gene_name,biotype,af_gtex,afc,afc_se,cs_id,cs_size,variant_id_gtex,skew_fc_vs_lead,lead_skew_sign
6585,chr1:16696772:C:T,chr1,16696772,C,T,sknsh,-0.701610,0.701610,EH38E2789727,0.083488,0.175940,0.005482,0.041362,0.068127,0.000778,0.030948,0.024138,0.064740,0.062500,0.023522,False,True,chr1:16696772:C:T,0.70161,0.951013,Pancreas,ENSG00000280114.1,ENSG00000280114,unprocessed_pseudogene,0.074586,1.461053,0.261062,3,1,1_16696772_C_T_b38,1.000000,-1
6534,chr1:16696631:C:G,chr1,16696631,C,G,sknsh,-0.790121,0.790121,EH38E2789727,0.227896,0.150908,0.203297,0.296960,0.221704,0.020062,0.267131,0.178082,0.264109,0.206731,0.302692,True,False,chr1:16696772:C:T,0.70161,0.951013,Pancreas,ENSG00000280114.1,ENSG00000280114,unprocessed_pseudogene,0.074586,1.461053,0.261062,3,1,1_16696631_C_G_b38,1.126155,-1
6562,chr1:16696689:A:G,chr1,16696689,A,G,sknsh,-0.785665,0.785665,EH38E2789727,0.006693,0.023426,0.000000,0.002684,0.000000,0.000000,0.000000,0.000000,0.000103,0.002846,0.000207,True,False,chr1:16696772:C:T,0.70161,0.951013,Pancreas,ENSG00000280114.1,ENSG00000280114,unprocessed_pseudogene,0.074586,1.461053,0.261062,3,1,1_16696689_A_G_b38,1.119803,-1
6519,chr1:16696590:C:G,chr1,16696590,C,G,sknsh,-1.225810,1.225810,EH38E2789727,0.000059,0.000000,0.000000,0.000000,0.000000,0.001735,0.000000,0.000000,0.000000,0.000000,0.000000,True,False,chr1:16696772:C:T,0.70161,0.951013,Pancreas,ENSG00000280114.1,ENSG00000280114,unprocessed_pseudogene,0.074586,1.461053,0.261062,3,1,1_16696590_C_G_b38,1.747140,-1
6515,chr1:16696584:C:G,chr1,16696584,C,G,sknsh,-1.291144,1.291144,EH38E2789727,0.000053,0.000000,0.000000,0.000065,0.000000,0.000000,0.000000,0.000000,0.000088,0.000000,0.000207,True,False,chr1:16696772:C:T,0.70161,0.951013,Pancreas,ENSG00000280114.1,ENSG00000280114,unprocessed_pseudogene,0.074586,1.461053,0.261062,3,1,1_16696584_C_G_b38,1.840259,-1
6521,chr1:16696604:C:T,chr1,16696604,C,T,sknsh,-0.734465,0.734465,EH38E2789727,0.000033,0.000000,0.000000,0.000131,0.000000,0.000193,0.000000,0.003401,0.000015,0.000000,0.000000,True,False,chr1:16696772:C:T,0.70161,0.951013,Pancreas,ENSG00000280114.1,ENSG00000280114,unprocessed_pseudogene,0.074586,1.461053,0.261062,3,1,1_16696604_C_T_b38,1.046829,-1
6518,chr1:16696590:C:T,chr1,16696590,C,T,sknsh,-1.060940,1.060940,EH38E2789727,0.000026,0.000024,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000044,0.000000,0.000000,True,False,chr1:16696772:C:T,0.70161,0.951013,Pancreas,ENSG00000280114.1,ENSG00000280114,unprocessed_pseudogene,0.074586,1.461053,0.261062,3,1,1_16696590_C_T_b38,1.512152,-1
6516,chr1:16696585:G:A,chr1,16696585,G,A,sknsh,-0.899341,0.899341,EH38E2789727,0.000020,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000029,0.000000,0.000207,True,False,chr1:16696772:C:T,0.70161,0.951013,Pancreas,ENSG00000280114.1,ENSG00000280114,unprocessed_pseudogene,0.074586,1.461053,0.261062,3,1,1_16696585_G_A_b38,1.281825,-1
6586,chr1:16696773:G:A,chr1,16696773,G,A,sknsh,-0.915378,0.915378,EH38E2789727,0.000007,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000208,True,False,chr1:16696772:C:T,0.70161,0.951013,Pancreas,ENSG00000280114.1,ENSG00000280114,unprocessed_pseudogene,0.074586,1.461053,0.261062,3,1,1_16696773_G_A_b38,1.304683,-1
6514,chr1:16696582:G:C,chr1,16696582,G,C,sknsh,-0.713434,0.713434,EH38E2789727,0.000007,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000015,0.000000,0.000000,True,False,chr1:16696772:C:T,0.70161,0.951013,Pancreas,ENSG00000280114.1,ENSG00000280114,unprocessed_pseudogene,0.074586,1.461053,0.261062,3,1,1_16696582_G_C_b38,1.016854,-1
